In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report, roc_auc_score


In [ ]:
df = pd.read_csv("BankChurners.csv")

print(df.shape)
df.head()

In [ ]:
df = df.drop(["CLIENTNUM"], axis=1)

In [ ]:
df = df.drop([
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1",
    "Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2"
], axis=1)

In [ ]:
df["Attrition_Flag"] = df["Attrition_Flag"].map({
    "Existing Customer": 0,
    "Attrited Customer": 1
})

In [ ]:
df = df.dropna()
df.info()

In [ ]:
df = pd.get_dummies(df, drop_first=True)
X = df.drop("Attrition_Flag", axis=1)
y = df["Attrition_Flag"]

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
sns.countplot(x='Attrition_Flag', data = df)
plt.title("Churn Distribution")
plt.xlabel("Churn (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['Customer_Age'], bins=20, kde=True)
plt.title("Customer Age Distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='Attrition_Flag', y='Months_on_book', data=df)
plt.title("Tenure vs Churn")
plt.xlabel("Churn")
plt.ylabel("Months on Book")
plt.show()

In [ ]:
sns.countplot(x='Months_Inactive_12_mon', hue='Attrition_Flag', data=df)
plt.title("Inactivity vs Churn")
plt.show()

In [ ]:
sns.countplot(x='Contacts_Count_12_mon', hue='Attrition_Flag', data=df)
plt.title("Customer Contacts vs Churn")
plt.show()

In [ ]:
## Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
log_model = LogisticRegression(max_iter=1000, class_weight='balanced')

log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)

In [ ]:
acc_log = accuracy_score(y_test, y_pred_log)

print("Logistic Accuracy:", acc_log)
print(classification_report(y_test, y_pred_log))

roc_log = roc_auc_score(y_test, log_model.predict_proba(X_test_scaled)[:,1])
print("Logistic ROC-AUC:", roc_log)

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)

print("Random Forest Accuracy:", acc_rf)
print(classification_report(y_test, y_pred_rf))

roc_rf = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1])
print("Random Forest ROC-AUC:", roc_rf)

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(
    kernel='rbf',
    C=1.0,
    gamma='scale',
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)
acc_svm = accuracy_score(y_test, y_pred_svm)

print("SVM Accuracy:", acc_svm)
print(classification_report(y_test, y_pred_svm))

roc_svm = roc_auc_score(y_test, svm_model.predict_proba(X_test_scaled)[:,1])
print("SVM ROC-AUC:", roc_svm)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# Logistic Regression
log_probs = log_model.predict_proba(X_test_scaled)[:,1]
fpr_log, tpr_log, _ = roc_curve(y_test, log_probs)

# Random Forest
rf_probs = rf_model.predict_proba(X_test)[:,1]
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

# SVM
svm_probs = svm_model.predict_proba(X_test_scaled)[:,1]
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_probs)

# Plot
plt.figure()

plt.plot(fpr_log, tpr_log, label="Logistic Regression")
plt.plot(fpr_rf, tpr_rf, label="Random Forest")
plt.plot(fpr_svm, tpr_svm, label="SVM")

plt.plot([0,1], [0,1], linestyle='--')  # baseline

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")

plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix

print("\nConfusion Matrix (Logistic Regression):")
print(confusion_matrix(y_test, y_pred_log))

print("\nConfusion Matrix (Random Forest):")
print(confusion_matrix(y_test, y_pred_rf))

print("\nConfusion Matrix (SVM):")
print(confusion_matrix(y_test, y_pred_svm))

In [ ]:
importance = pd.Series(rf_model.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print("\nTop 10 Important Features:")
print(importance.head(10))

# Plot
importance.head(10).plot(kind='bar')
plt.title("Top Features Affecting Churn")
plt.show()

In [ ]:
def prepare_input(input_dict):
    df_input = pd.DataFrame([input_dict])
    df_input = pd.get_dummies(df_input)

    # Align columns
    df_input = df_input.reindex(columns=X.columns, fill_value=0)

    return df_input

In [ ]:
def predict_dynamic(input_dict):

    # Prepare input
    input_df = pd.DataFrame([input_dict])
    input_df = pd.get_dummies(input_df)
    input_df = input_df.reindex(columns=X.columns, fill_value=0)

    # Scale for Logistic & SVM
    input_scaled = scaler.transform(input_df)

    # Logistic Regression
    log_prob = log_model.predict_proba(input_scaled)[0][1]
    log_pred = 1 if log_prob >= 0.5 else 0

    # Random Forest
    rf_prob = rf_model.predict_proba(input_df)[0][1]
    rf_pred = 1 if rf_prob >= 0.5 else 0

    # SVM
    svm_prob = svm_model.predict_proba(input_scaled)[0][1]
    svm_pred = 1 if svm_prob >= 0.5 else 0

    # Labels
    log_label = "Yes" if log_pred else "No"
    rf_label = "Yes" if rf_pred else "No"
    svm_label = "Yes" if svm_pred else "No"

    # ?? Confidence-based decision among ALL 3 models
    log_conf = abs(log_prob - 0.5)
    rf_conf = abs(rf_prob - 0.5)
    svm_conf = abs(svm_prob - 0.5)

    if rf_conf > log_conf and rf_conf > svm_conf:
        final_model = "Random Forest"
        final_label = rf_label
        final_prob = rf_prob
    elif svm_conf > log_conf and svm_conf > rf_conf:
        final_model = "SVM"
        final_label = svm_label
        final_prob = svm_prob
    else:
        final_model = "Logistic Regression"
        final_label = log_label
        final_prob = log_prob

    # Output
    print("\nPrediction output:")
    print(f"Logistic Regression -> Churn: {log_label}, Probability: {log_prob*100:.2f}%")
    print(f"Random Forest       -> Churn: {rf_label}, Probability: {rf_prob*100:.2f}%")
    print(f"SVM                 -> Churn: {svm_label}, Probability: {svm_prob*100:.2f}%")

    print(f"\nFinal ({final_model}) -> Churn: {final_label}, Probability: {final_prob*100:.2f}%")

In [ ]:
customer_low = {
    "Customer_Age": 60,
    "Dependent_count": 3,
    "Months_on_book": 72,
    "Total_Relationship_Count": 6,
    "Months_Inactive_12_mon": 0,
    "Contacts_Count_12_mon": 0,
    "Credit_Limit": 20000,
    "Total_Revolving_Bal": 2000,
    "Avg_Open_To_Buy": 18000,
    "Total_Amt_Chng_Q4_Q1": 1.3,
    "Total_Trans_Amt": 4000,
    "Total_Trans_Ct": 60,
    "Total_Ct_Chng_Q4_Q1": 1.5,
    "Avg_Utilization_Ratio": 0.2
}
predict_dynamic(customer_low)

In [ ]:
customer_high = {
    "Customer_Age": 40,
    "Dependent_count": 1,
    "Months_on_book": 6,
    "Total_Relationship_Count": 1,
    "Months_Inactive_12_mon": 5,
    "Contacts_Count_12_mon": 5,
    "Credit_Limit": 3000,
    "Total_Revolving_Bal": 2800,
    "Avg_Open_To_Buy": 200,
    "Total_Amt_Chng_Q4_Q1": 0.5,
    "Total_Trans_Amt": 700,
    "Total_Trans_Ct": 12,
    "Total_Ct_Chng_Q4_Q1": 0.4,
    "Avg_Utilization_Ratio": 0.9
}
predict_dynamic(customer_high)

In [ ]:
customer_mid = {
    "Customer_Age": 50,
    "Dependent_count": 2,
    "Months_on_book": 24,
    "Total_Relationship_Count": 3,
    "Months_Inactive_12_mon": 2,
    "Contacts_Count_12_mon": 2,
    "Credit_Limit": 8000,
    "Total_Revolving_Bal": 4000,
    "Avg_Open_To_Buy": 4000,
    "Total_Amt_Chng_Q4_Q1": 0.9,
    "Total_Trans_Amt": 1500,
    "Total_Trans_Ct": 30,
    "Total_Ct_Chng_Q4_Q1": 0.9,
    "Avg_Utilization_Ratio": 0.5
}

predict_dynamic(customer_mid)

In [ ]:
customer_o = {
    "Customer_Age": 45,
    "Dependent_count": 2,
    "Months_on_book": 42,          
    "Total_Relationship_Count": 4, # stabilize RF ?
    "Months_Inactive_12_mon": 3,   
    "Contacts_Count_12_mon": 3,    
    "Credit_Limit": 10000,
    "Total_Revolving_Bal": 5500,   # moderate-high
    "Avg_Open_To_Buy": 4500,
    "Total_Amt_Chng_Q4_Q1": 0.9,
    "Total_Trans_Amt": 1800,
    "Total_Trans_Ct": 30,
    "Total_Ct_Chng_Q4_Q1": 0.9,
    "Avg_Utilization_Ratio": 0.65  # tuned value
}
predict_dynamic(customer_o)